In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SelectKBest, f_regression
import warnings
warnings.filterwarnings('ignore')


In [8]:
df = pd.read_csv(r"C:\\Users\\zizou\\OneDrive\\Desktop\\stage 3ème\\day 2\\csvfiles\\dataframefinale.csv", sep=';', encoding='utf-8-sig')

# Define features and target
features = ['Nombre de Titres', 'Echéance', 'Taux', 'Jour', 'Mois', 'NumeroSemaine', 'Trimestre', 'JourSemaineNum']
X = df[features]
y = df['Montant']

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Function to calculate RMSE
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [9]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'SVR': SVR()
}

In [10]:
results = []
print("=== Step 1: Initial Training and RMSE ===")
for name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)
    # Predict on test set
    y_pred = model.predict(X_test)
    # Calculate RMSE
    rmse = calculate_rmse(y_test, y_pred)
    results.append({'Model': name, 'Step': 'Initial', 'RMSE': rmse})
    print(f"{name} RMSE: {rmse:.4f}")

=== Step 1: Initial Training and RMSE ===
Linear Regression RMSE: 7.0479
Ridge Regression RMSE: 7.0479
Lasso Regression RMSE: 7.0584
Decision Tree RMSE: 4.9810
Random Forest RMSE: 3.9204
SVR RMSE: 7.3261


In [11]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n=== Step 2: Training with Normalization and RMSE ===")
for name, model in models.items():
    # Train the model on scaled data
    model.fit(X_train_scaled, y_train)
    # Predict on test set
    y_pred = model.predict(X_test_scaled)
    # Calculate RMSE
    rmse = calculate_rmse(y_test, y_pred)
    results.append({'Model': name, 'Step': 'Normalized', 'RMSE': rmse})
    print(f"{name} RMSE (Normalized): {rmse:.4f}")


=== Step 2: Training with Normalization and RMSE ===
Linear Regression RMSE (Normalized): 7.0479
Ridge Regression RMSE (Normalized): 7.0480
Lasso Regression RMSE (Normalized): 7.2257
Decision Tree RMSE (Normalized): 4.9657
Random Forest RMSE (Normalized): 3.9228
SVR RMSE (Normalized): 6.8887


In [12]:
selector = SelectKBest(score_func=f_regression, k=5)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

# Get selected feature names
selected_features = X_train.columns[selector.get_support()].tolist()
print("\n=== Step 3: Selected Features ===")
print(f"Selected Features: {selected_features}")

for name, model in models.items():
    # Train the model on selected features
    model.fit(X_train_selected, y_train)
    # Predict on test set
    y_pred = model.predict(X_test_selected)
    # Calculate RMSE
    rmse = calculate_rmse(y_test, y_pred)
    results.append({'Model': name, 'Step': 'Feature Selection', 'RMSE': rmse})
    print(f"{name} RMSE (Feature Selection): {rmse:.4f}")


=== Step 3: Selected Features ===
Selected Features: ['Nombre de Titres', 'Taux', 'Jour', 'Trimestre', 'JourSemaineNum']
Linear Regression RMSE (Feature Selection): 7.0567
Ridge Regression RMSE (Feature Selection): 7.0567
Lasso Regression RMSE (Feature Selection): 7.0642
Decision Tree RMSE (Feature Selection): 5.1824
Random Forest RMSE (Feature Selection): 4.2975
SVR RMSE (Feature Selection): 7.3274


In [13]:
param_grids = {
    'Linear Regression': {},
    'Ridge Regression': {'alpha': [0.1, 1.0, 10.0, 100.0]},
    'Lasso Regression': {'alpha': [0.001, 0.01, 0.1, 1.0]},
    'Decision Tree': {'max_depth': [None, 10, 20], 'min_samples_split': [2, 5, 10]},
    'Random Forest': {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20]},
    'SVR': {'C': [0.1, 1, 10], 'epsilon': [0.01, 0.1, 0.5]}
}

print("\n=== Step 4: Hyperparameter Tuning and RMSE ===")
for name, model in models.items():
    if param_grids[name]:  # Skip if no hyperparameters to tune
        grid_search = GridSearchCV(model, param_grids[name], cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
        grid_search.fit(X_train_selected, y_train)
        best_model = grid_search.best_estimator_
        print(f"{name} Best Params: {grid_search.best_params_}")
    else:
        best_model = model
        best_model.fit(X_train_selected, y_train)
    
    # Predict on test set
    y_pred = best_model.predict(X_test_selected)
    # Calculate RMSE
    rmse = calculate_rmse(y_test, y_pred)
    results.append({'Model': name, 'Step': 'Tuned', 'RMSE': rmse})
    print(f"{name} RMSE (Tuned): {rmse:.4f}")


=== Step 4: Hyperparameter Tuning and RMSE ===
Linear Regression RMSE (Tuned): 7.0567
Ridge Regression Best Params: {'alpha': 100.0}
Ridge Regression RMSE (Tuned): 7.0566
Lasso Regression Best Params: {'alpha': 0.001}
Lasso Regression RMSE (Tuned): 7.0567
Decision Tree Best Params: {'max_depth': None, 'min_samples_split': 10}
Decision Tree RMSE (Tuned): 5.0239
Random Forest Best Params: {'max_depth': 20, 'n_estimators': 200}
Random Forest RMSE (Tuned): 4.2540
SVR Best Params: {'C': 1, 'epsilon': 0.5}
SVR RMSE (Tuned): 7.2855


In [14]:
results_df = pd.DataFrame(results)
print("\n=== Summary of RMSE Results ===")
print(results_df.pivot(index='Model', columns='Step', values='RMSE'))


=== Summary of RMSE Results ===
Step               Feature Selection   Initial  Normalized     Tuned
Model                                                               
Decision Tree               5.182381  4.980985    4.965724  5.023932
Lasso Regression            7.064226  7.058381    7.225735  7.056652
Linear Regression           7.056675  7.047944    7.047944  7.056675
Random Forest               4.297497  3.920395    3.922844  4.253960
Ridge Regression            7.056675  7.047938    7.047958  7.056632
SVR                         7.327367  7.326092    6.888717  7.285453


In [15]:
results_df.to_csv(r"C:\\Users\\zizou\\OneDrive\\Desktop\\stage 3ème\\day 2\\csvfiles\\modeling_results.csv", index=False, sep=';', encoding='utf-8-sig')
print("\n✅ Results saved to modeling_results.csv")


✅ Results saved to modeling_results.csv
